<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-02/AI_Modul_2.1_Praktikum_Konsep_Algoritma.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 2.1: Praktikum Konsep Algoritma
### Benchmarking Asimtotik Big-O: Linear vs Binary Search, Analisis Pengurutan Sensor, dan Logika Kontrol AI Perkebunan

---

> **Diktat Terkait:** [AI_Modul_2.1_Konsep_Algoritma.md](../../docs/part-02/AI_Modul_2.1_Konsep_Algoritma.md)  
> **Outputs:** Kode program pengujian kompleksitas waktu asimtotik dengan pencatat waktu presisi mikrodetik, komparasi Bubble Sort vs TimSort, dan implementasi mesin aturan kontrol irigasi cerdas multi-kondisi.  
> **Outcomes:** Mahasiswa terampil mengukur efisiensi algoritma sebelum dijalankan pada sistem riil, memilih algoritma bernotasi efisien ($\mathcal{O}(\log N)$ daripada $\mathcal{O}(N)$), dan merancang struktur kontrol logika yang kokoh.  
> **Impacts:** Penghematan daya komputasi dan memori pada perangkat *Edge AI* (drone dan IoT perkebunan) serta jaminan keandalan eksekusi sistem industri waktu-nyata.
>
> **Tujuan Pembelajaran:**  
> 1. Mengukur perbedaan jumlah langkah dan waktu eksekusi riil antara Linear Search $\mathcal{O}(N)$ dan Binary Search $\mathcal{O}(\log N)$ pada katalog $1.000.000$ bibit sawit.  
> 2. Mengamati lonjakan waktu kuadratik $\mathcal{O}(N^2)$ pada algoritma Bubble Sort dibandingkan pengurutan kuasilinier $\mathcal{O}(N \log N)$.  
> 3. Membangun mesin logika kontrol sekuensial, percabangan, dan perulangan untuk sistem otomatisasi sensor agro-industri.

## 1. Persiapan Environment & Import Library
Jalankan sel berikut untuk memuat pustaka pengukur waktu, manipulasi numerik, dan visualisasi grafik:

In [ ]:
# Mengimpor modul sys untuk membaca metadata interpreter Python
import sys

# Mengimpor modul time untuk mencatat waktu eksekusi mikrodetik
import time

# Mengimpor pustaka numpy untuk operasi vektor dan pembangkitan data sintetis
import numpy as np

# Mengimpor pustaka matplotlib untuk membuat visualisasi kurva kompleksitas
import matplotlib.pyplot as plt

# Menetapkan nilai seed acak agar hasil simulasi deterministik dan konsisten
np.random.seed(42)

# Mengatur konfigurasi gaya grafik visualisasi
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120

# Menampilkan informasi versi Python
print(f"Python Version : {sys.version.split()[0]}")
print("[OK] Seluruh modul saintifik berhasil dimuat dengan sukses!")

## 2. Eksperimen 1: Pencarian Data (Linear Search vs Binary Search)

Kita menguji basis data katalog **$N = 1.000.000$ bibit kelapa sawit terdaftar**.
Target diletakkan di indeks paling akhir untuk menguji skenario terburuk (*worst-case scenario*):
* **Linear Search**: Memeriksa elemen satu per satu dari awal $\to \mathcal{O}(N)$.
* **Binary Search**: Membagi ruang pencarian menjadi $50\%$ pada setiap perulangan $\to \mathcal{O}(\log_2 N)$.

In [ ]:
# Membangkitkan katalog 1.000.000 ID bibit sawit terurut secara sintetis
ukuran_katalog = 1_000_000
katalog_bibit = np.arange(100_000, 100_000 + ukuran_katalog)

# Menetapkan target ID bibit di indeks paling akhir (worst-case)
target_id = katalog_bibit[-1]

# Implementasi Algoritma Linear Search O(N)
def linear_search(array_data, target):
    langkah = 0
    for indeks in range(len(array_data)):
        langkah += 1
        if array_data[indeks] == target:
            return indeks, langkah
    return -1, langkah

# Implementasi Algoritma Binary Search O(log N)
def binary_search(array_data, target):
    langkah = 0
    kiri = 0
    kanan = len(array_data) - 1
    
    while kiri <= kanan:
        langkah += 1
        tengah = (kiri + kanan) // 2
        if array_data[tengah] == target:
            return tengah, langkah
        elif array_data[tengah] < target:
            kiri = tengah + 1
        else:
            kanan = tengah - 1
    return -1, langkah

# Mengukur eksekusi Linear Search
t_start = time.perf_counter()
pos_lin, steps_lin = linear_search(katalog_bibit, target_id)
durasi_lin = (time.perf_counter() - t_start) * 1000.0 # ms

# Mengukur eksekusi Binary Search
t_start = time.perf_counter()
pos_bin, steps_bin = binary_search(katalog_bibit, target_id)
durasi_bin = (time.perf_counter() - t_start) * 1000.0 # ms

print("=" * 65)
print("HASIL BENCHMARK PENCARIAN DATA (N = 1.000.000 ENTITAS)")
print("=" * 65)
print(f"-> Linear Search O(N)     : {steps_lin:,} langkah | Waktu: {durasi_lin:.3f} ms")
print(f"-> Binary Search O(log N) : {steps_bin:,} langkah      | Waktu: {durasi_bin:.4f} ms")
print(f"-> Efisiensi Langkah      : Binary Search {steps_lin // steps_bin:,}x lebih hemat operasi!")
print("=" * 65)

## 3. Eksperimen 2: Pengurutan Sensor (Bubble Sort vs TimSort)

Kita mengurutkan $N = 2.000$ sampel suhu kebun acak:
* **Bubble Sort**: Dua loop bersarang $\to \mathcal{O}(N^2)$.
* **Python TimSort**: Algoritma hibrida adaptif $\to \mathcal{O}(N \log N)$.

In [ ]:
# Membangkitkan 2.000 log suhu tanah kebun acak
N_sensor = 2000
log_suhu = np.random.uniform(22.0, 38.0, size=N_sensor).tolist()

# Implementasi Bubble Sort O(N^2)
def bubble_sort(arr_input):
    arr = list(arr_input)
    n = len(arr)
    for i in range(n):
        for j in range(0, n - i - 1):
            if arr[j] > arr[j + 1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
    return arr

# Mengukur eksekusi Bubble Sort
t_start = time.perf_counter()
hasil_bubble = bubble_sort(log_suhu)
durasi_bubble = time.perf_counter() - t_start

# Mengukur eksekusi TimSort bawaan Python
t_start = time.perf_counter()
hasil_timsort = sorted(log_suhu)
durasi_timsort = time.perf_counter() - t_start

print("=" * 65)
print(f"HASIL BENCHMARK PENGURUTAN DATA SENSOR (N = {N_sensor:,} DATA)")
print("=" * 65)
print(f"-> Bubble Sort O(N^2)     : {durasi_bubble:.4f} detik")
print(f"-> Python TimSort O(NlogN): {durasi_timsort:.6f} detik")
print(f"-> Kecepatan Relatif      : TimSort ~{durasi_bubble / max(durasi_timsort, 1e-6):.1f}x lebih cepat!")
print("=" * 65)

## 4. Eksperimen 3: Implementasi Logika Struktur Kontrol Cerdas (Smart Irrigation)

Sistem otomasi irigasi cerdas memadukan:
1. **Sekuensial**: Membaca parameter sensor secara teratur.
2. **Percabangan**: Mengevaluasi ambang batas kritis (Kelembaban Tanah, Suhu Udara, dan Radiasi Surya).
3. **Perulangan**: Memproses antrian aliran data (*stream*) multi-titik blok kebun.

In [ ]:
# Simulasi data streaming multi-sensor kebun (Kelembaban %, Suhu °C, Radiasi W/m2)
sensor_stream = [
    {"titik": "Blok-A1", "kelembaban": 55.0, "suhu": 29.0, "radiasi": 450},
    {"titik": "Blok-A2", "kelembaban": 38.0, "suhu": 34.5, "radiasi": 780}, # Kering & Panas
    {"titik": "Blok-A3", "kelembaban": 42.0, "suhu": 31.0, "radiasi": 520}, # Moderat Kering
    {"titik": "Blok-A4", "kelembaban": 70.0, "suhu": 26.0, "radiasi": 200}, # Lembab Aman
]

print("=" * 65)
print("LOGIKA KONTROL CERDAS IRIGASI MULTI-TITIK BLOK KEBUN")
print("=" * 65)

# Struktur Perulangan (Looping)
for data in sensor_stream:
    # Struktur Sekuensial (Sequence)
    lokasi = data["titik"]
    rh = data["kelembaban"]
    t = data["suhu"]
    rad = data["radiasi"]
    
    # Struktur Percabangan (Branching)
    if rh < 40.0 and t > 32.0:
        status_aksi = "IRIGASI MAKSIMAL: Buka Katup 100% (Darurat Stres Air)"
        kode_warna = "[BAHAYA]"
    elif rh < 45.0 or rad > 700:
        status_aksi = "IRIGASI SEDANG  : Buka Katup 50% (Pencegahan Dehidrasi)"
        kode_warna = "[WASPADA]"
    else:
        status_aksi = "STANDBY HEMAT   : Katup Tertutup (Kondisi Tanah Lembab)"
        kode_warna = "[NORMAL]"
        
    print(f"-> {lokasi} | RH: {rh:.1f}%, Suhu: {t:.1f} C | {kode_warna:<9} -> {status_aksi}")
print("=" * 65)

## 5. Visualisasi Kurva Pertumbuhan Kompleksitas Asimtotik

In [ ]:
# Visualisasi kurva laju pertumbuhan Big-O
n_vals = np.linspace(1, 20, 300)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(n_vals, np.log2(n_vals), label='O(log N) - Logaritmik (Binary Search)', color='green', linewidth=2)
ax.plot(n_vals, n_vals, label='O(N) - Linier (Linear Search)', color='orange', linewidth=2)
ax.plot(n_vals, n_vals * np.log2(n_vals), label='O(N log N) - Kuasilinier (TimSort)', color='blue', linewidth=2)
ax.plot(n_vals, n_vals**2, label='O(N^2) - Kuadratik (Bubble Sort)', color='red', linewidth=2)

ax.set_ylim(0, 100)
ax.set_xlim(1, 15)
ax.set_title("Perbandingan Laju Pertumbuhan Kompleksitas Algoritma (Big-O)", fontsize=12, fontweight='bold')
ax.set_xlabel("Ukuran Data Masukan (N)", fontsize=10, fontweight='bold')
ax.set_ylabel("Jumlah Operasi Komputasi", fontsize=10, fontweight='bold')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

## 6. Tugas Mandiri & Eksplorasi Mahasiswa

1. **Eksplorasi Skenario Terbaik (*Best-Case*)**: Modifikasi target pencarian Binary Search menjadi target yang berada tepat di tengah array (`target = katalog_bibit[len(katalog_bibit)//2]`). Berapa langkah yang dibutuhkan? Mengapa waktu eksekusinya menjadi $\mathcal{O}(1)$?
2. **Skalabilitas Data Sensor**: Ujilah Bubble Sort dengan $N = 5.000$ sampel. Berapa detik waktu yang dibutuhkan? Bandingkan dengan $N = 2.000$ dan buktikan apakah peningkatannya mendekati rasio kuadrat $(5000/2000)^2 = 6.25\times$!